# Validation Strategy and Data Leakage

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Import datasets

In [2]:
from sklearn.datasets import load_breast_cancer

## Import Scikit learn Modules

In [3]:
from sklearn.model_selection import (
     train_test_split,GroupShuffleSplit,TimeSeriesSplit
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
     accuracy_score,
     precision_score,
     recall_score,
     f1_score,
     classification_report,
     confusion_matrix
)

## Basic Stratified Split

### Load Dataset

In [7]:
df=load_breast_cancer(as_frame=True)
x=df.data
y=df.target
print(x.shape)
print(y.shape)

print("\nClass distribution:")
print(y.value_counts(normalize=True))

(569, 30)
(569,)

Class distribution:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64


## Stratified Train/Validation/test Split

### Temperory Train and Test

In [10]:
X_train_temp,X_test,y_train_temp,y_test=train_test_split(
     x,y,
     test_size=0.2,
     stratify=y,
     random_state=42
)

### Train and Validataion

In [14]:
X_train,X_val,y_train,y_val=train_test_split(
     X_train_temp,
     y_train_temp,
     test_size=0.25,
     stratify=y_train_temp,
     random_state=42
)

### Outputs

In [16]:
print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation class distribution:")
print(y_val.value_counts(normalize=True))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Train size: (341, 30)
Validation size: (114, 30)
Test size: (114, 30)

Train class distribution:
target
1    0.627566
0    0.372434
Name: proportion, dtype: float64

Validation class distribution:
target
1    0.622807
0    0.377193
Name: proportion, dtype: float64

Test class distribution:
target
1    0.622807
0    0.377193
Name: proportion, dtype: float64


## Safe Pipeline Evaluation

### Build Leakage-Safe Pipeline

In [17]:
safe_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])
safe_pipeline.fit(X_train, y_train)
val_pred = safe_pipeline.predict(X_val)
print("Validation Results")
print("Accuracy :", accuracy_score(y_val, val_pred))
print("Precision:", precision_score(y_val, val_pred))
print("Recall   :", recall_score(y_val, val_pred))
print("F1       :", f1_score(y_val, val_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))
print("\nClassification Report:")
print(classification_report(y_val, val_pred))

Validation Results
Accuracy : 0.9912280701754386
Precision: 0.9861111111111112
Recall   : 1.0
F1       : 0.993006993006993

Confusion Matrix:
[[42  1]
 [ 0 71]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        43
           1       0.99      1.00      0.99        71

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



### Final Test Evaluation

In [18]:
test_pred = safe_pipeline.predict(X_test)
print("Final Test Results")
print("Accuracy :", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall   :", recall_score(y_test, test_pred))
print("F1       :", f1_score(y_test, test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print("\nClassification Report:")
print(classification_report(y_test, test_pred))

Final Test Results
Accuracy : 0.9912280701754386
Precision: 0.9861111111111112
Recall   : 1.0
F1       : 0.993006993006993

Confusion Matrix:
[[42  1]
 [ 0 71]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        43
           1       0.99      1.00      0.99        71

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



## leaky vs Safe Preprocessing Example

### Wrong Scaling before split

In [21]:
scaler=StandardScaler()
x_scaled_w=scaler.fit_transform(x)
X_train_w,X_test_w,y_train_w,y_test_w=train_test_split(
     x_scaled_w,y,
     test_size=0.2,
     stratify=y,
     random_state=42
)
wrong_model = LogisticRegression(max_iter=5000)
wrong_model.fit(X_train_w, y_train_w)

wrong_pred = wrong_model.predict(X_test_w)

print("Leaky preprocessing result:")
print("Accuracy:", accuracy_score(y_test_w, wrong_pred))

Leaky preprocessing result:
Accuracy: 0.9824561403508771
